# Manual runs of VQC on AWS devices
### Note:
1. For the manual runs control values are set in modules.config
2. It also possible to run automatically

Import modules needed:

In [ ]:
import numpy as np
import copy
import time
from pathlib import Path

from modules.helper_functions_tsp import (
        find_problem_size, 
        find_run_stats,
        find_distances_array,
        detect_quantum_GPU_support, 
        )

from modules.AWS_helper_functions_tsp import (
        define_parameters, 
        cost_fn_fact, 
        update_parameters_using_gradient, 
        create_initial_rotations,
        cost_func_evaluate, 
        )

from modules.helper_functions_quantum import (
    bind_weights,
    vqc_circuit,
)

from modules.graph_functions import cost_graph_multi

from classes.MyDataLogger import MyDataLogger, MySubDataLogger

from modules.config import(
        TARGET,
        AWS
        )               

Set random seed so results are reproducible for testing.

In [2]:
np.random.seed(42)

Import constants needed.  Note that all are read directly from modules.config:

In [3]:
from modules.config import SLICES

CHANGE_EACH_PARAMETER = False       # Iterate through each parameter in the circuit
PLOT_PARAMETER_EVALUATION = False   # Plot the evaluation of each parameter         

In [4]:
if TARGET not in  ['local_aws', 'local_aws_test','local_qiskit']:
    raise Exception(f'Notebook is only written for AWS to be run locally and {TARGET=}')

In [5]:
from modules.AWS_helper_functions_tsp import find_device_string, find_device
device = find_device(TARGET)

In [6]:
if not AWS:
    raise Exception(f'Notebook is only written for AWS.  Here {AWS=}')

In [7]:
if detect_quantum_GPU_support():
    print('Quantum GPU support detected')
else:
    print('Quantum GPU support not detected')

Quantum GPU support not detected


Instantiate data logger - this will create a folder for graphs and results:


In [8]:
datalogger = MyDataLogger()
sdl = MySubDataLogger(runid = datalogger.runid)
sdl.update_general_constants_from_config()
sdl.update_quantum_constants_from_config()
sdl.aws = True
sdl.validate_input()

SubDataLogger instantiated.  Run ID = 20260516-16-55-41 - 16-55-41


From the number of locations visited find the qubits and longest binary string.

In [9]:
sdl.qubits = find_problem_size(sdl)
print(f'There are {sdl.qubits} logical qubits needed for {sdl.locations} locations in the {sdl.formulation} formulation.')
print(f'Running mode {sdl.mode} with {sdl.shots} shots')

There are 3 logical qubits needed for 4 locations in the original formulation.
Running mode 13 with 1024 shots


Data sources are held locally to avoid downstream dependencies.  Read the data, and print out the filename and best distance held in the data.  Read and validate the distance array.  This checks the array is the correct shape, and is symmetric.

In [10]:
distance_array, best_dist = find_distances_array(sdl.locations, print_comments=True)

Reading distance data
Data will be read from filename networks\four_d.txt.
It is known that the shortest distance is 21


Define the VQC circuits with appropriate parameters, and draw the circuit.

In [11]:
sdl.num_params = sdl.calculate_parameter_numbers()
print(f'Number of parameters to be optimized is {sdl.num_params}')
params = define_parameters(sdl.qubits,
                           sdl.mode,
                           sdl.num_params,
                          )

print(f'Parameters are: {params}')
qc = vqc_circuit(
    qubits=sdl.qubits,
    mode=sdl.mode,
    #noise=sdl.noise,
    layers=sdl.layers,
    params=params,
    target=TARGET,)

filename = Path.joinpath(datalogger.graph_sub_path, f'initial_unbound_circuit{sdl.mode}.pdf')

if sdl.qubits < 15:
    print(qc)

Number of parameters to be optimized is 6
Parameters are: [param_0, param_1, param_2, param_3, param_4, param_5]
qubit_dict={0: 0, 1: 1, 2: 2}, qubits_measured=3, qubits=3
After circuit set up, the verbatim box receives the following circuitQubitSet([Qubit(0), Qubit(1), Qubit(2)])
The qubit dictionary is {0: 0, 1: 1, 2: 2}
After measurement, the following qubits are measured [0, 1, 2]
T  : │        0        │     1      │     2      │     3      │       4       │  5  │     6      │     7      │     8      │       9       │ 10  │      11       │     12     │     13     │      14       │ 15  │      16       │     17     │     18     │     19     │      20       │ 21  │
                        ┌──────────┐ ┌──────────┐ ┌──────────┐ ┌─────────────┐                                                                                                                              ┌───┐ ┌─────────────┐ ┌──────────┐ ┌──────────┐ ┌──────────┐                 ┌───┐ 
q0 : ───StartVerbatim───┤ Rz(1.57) ├

Determine the cost function.  This receives a binary list and returns a real value:

In [12]:
cost_fn = cost_fn_fact(
    locations=sdl.locations,
    qubits=sdl.qubits,
    gray=sdl.gray,
    formulation=sdl.formulation,
    distance_array=distance_array,
    target=TARGET
                       )

Create random initial rotations, and bind these to the parameters in a new circuit `bc`.

In [13]:
bin_hot_start_list=[]
if sdl.hot_start:
    raise Exception(f'The hot start list cannot be run at the moment with Amazon.  {sdl.hot_start=}')
init_rots = create_initial_rotations(sdl.qubits,
                                     sdl.num_params,
                                     sdl.mode,
                                     sdl.layers,
                                     sdl.hot_start,
                                     bin_hot_start_list,
                                    )

print(f'The initial parameters (weights) are {init_rots}')
bc = bind_weights(
    params=params, 
    rots=init_rots, 
    qc=qc,
    target=TARGET,
    )
filename = Path.joinpath(datalogger.graph_sub_path, f'initial_bound_circuit{sdl.mode}.pdf')
if sdl.qubits < 15:
    print(bc)

The initial parameters (weights) are [1.75510165 2.30002029 4.28417018 4.29638061 2.9676893  2.46494103]
T  : │        0        │     1      │     2      │     3      │     4      │  5  │     6      │     7      │     8      │     9      │ 10  │     11     │     12     │     13     │     14     │ 15  │     16     │     17     │     18     │     19     │      20       │ 21  │
                        ┌──────────┐ ┌──────────┐ ┌──────────┐ ┌──────────┐                                                                                                                     ┌───┐ ┌──────────┐ ┌──────────┐ ┌──────────┐ ┌──────────┐                 ┌───┐ 
q0 : ───StartVerbatim───┤ Rz(1.57) ├─┤ Rx(1.57) ├─┤ Rz(1.57) ├─┤ Rz(1.76) ├───●─────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤ Z ├─┤ Rz(4.30) ├─┤ Rz(1.57) ├─┤ Rx(1.57) ├─┤ Rz(1.57) ├───EndVerbatim───┤ M ├─
              ║         └──────────┘ └──────────┘ └──────────┘ └──────────┘ 

In [14]:
print([p.name for p in qc.parameters])
print([str(p) for p in params])

['param_4', 'param_0', 'param_2', 'param_5', 'param_3', 'param_1']
['param_0', 'param_1', 'param_2', 'param_3', 'param_4', 'param_5']


Find average and lowest cost for the starting circuit, and print these:

In [15]:
cost_start, lowest_to_date, _ = cost_func_evaluate(noise=sdl.noise,
                                                   shots= sdl.shots,
                                                   cost_fn=cost_fn, 
                                                   model=bc, 
                                                   target=TARGET,
                                                   average_slice=SLICES[0], 
                                                   )
print(f'For the starting circuit the average cost is {cost_start:.1f} and the lowest cost is {lowest_to_date:.1f}')

For the starting circuit the average cost is 21.4 and the lowest cost is 21.0


In [16]:
cost_start, lowest_to_date, _ = cost_func_evaluate(noise=sdl.noise,
                                                   shots= sdl.shots,
                                                   cost_fn=cost_fn, 
                                                   model=bc, 
                                                   target=TARGET,
                                                   average_slice=SLICES[0], 
                                                   )
print(f'For the starting circuit the average cost is {cost_start:.1f} and the lowest cost is {lowest_to_date:.1f}')

For the starting circuit the average cost is 21.2 and the lowest cost is 21.0


Do a run for each slice, time, and save results.

In [17]:
rots = copy.deepcopy(init_rots)
av_cost_list_all, lowest_list_all, sliced_cost_list_all = [], [], []
sdl_list = []
for i, slice in enumerate(SLICES):
    t0 = time.time()
    sdl_list.append(MySubDataLogger(runid=datalogger.runid))
    sdl_list[i].update_general_constants_from_config()
    sdl_list[i].update_quantum_constants_from_config()
    sdl_list[i].slice = slice
    sdl_list[i].qubits = sdl.qubits
    
    new_output_data = update_parameters_using_gradient(sdl_list[i].locations,
                                                       sdl_list[i].qubits,
                                                       sdl_list[i].slice,
                                                       sdl_list[i].shots,
                                                       sdl_list[i].mode,
                                                       sdl_list[i].iterations,
                                                       sdl_list[i].gray,
                                                       sdl_list[i].hot_start,
                                                       sdl_list[i].gradient_type,
                                                       sdl_list[i].formulation,
                                                       sdl_list[i].layers,
                                                       sdl_list[i].alpha,
                                                       sdl_list[i].big_a,
                                                       sdl_list[i].c,
                                                       sdl_list[i].eta,
                                                       sdl_list[i].gamma,
                                                       sdl_list[i].s,
                                                       sdl_list[i].noise,
                                                       params=params,
                                                       rots=init_rots,
                                                       cost_fn=cost_fn,
                                                       qc=qc,
                                                       target=TARGET,
                                                       )
                        
    sdl_list[i].index_list, sdl_list[i].sliced_list, sdl_list[i].lowest_list, _ , sdl_list[i].average_list, _ = new_output_data
    
    av_cost_list_all.append(sdl_list[i].average_list)
    lowest_list_all.append(sdl_list[i].lowest_list)
    sliced_cost_list_all.append(sdl_list[i].sliced_list)

    best_dist_found, iteration_found = find_run_stats(sdl_list[i].lowest_list)
    print(f'With a slice of {slice} the best distance found was {best_dist_found} at iteration {iteration_found }')
    print(f'This is compared to the best distance of {best_dist}, giving a quality of {best_dist/best_dist_found:.1%}')
    #if sdl_list[i].hot_start:
    #    sdl_list[i].hot_start_dist = hot_start_distance
    #else:
        #sdl_list[i].hot_start_dist = 'n/a'
    sdl_list[i].hot_start_dist = 'n/a'
    sdl_list[i].best_dist_found = best_dist_found
    sdl_list[i].best_dist = best_dist
    sdl_list[i].iteration_found = iteration_found

    t1 = time.time()
    elapsed = t1-t0
    print(f'The time taken to run the code is {elapsed:.3f} seconds')
    sdl_list[i].elapsed = elapsed
    sdl_list[i].update_cache_statistics(cost_fn)
    sdl_list[i].save_results_to_csv()
    sdl_list[i].save_detailed_results()

SubDataLogger instantiated.  Run ID = 20260516-16-55-41 - 16-55-41
For iteration 0 using the best 80.0 percent of the results
The average cost from the sample is 22.094 and the top-sliced average of the best results is 21.338
The lowest cost from the sample is 21.000, flush=True
The lowest cost to date is 21.000 corresponding to bit string [0, 0, 1]
and route [0, 1, 3, 2]
Metrics - timestamp=1778946941.497345; average_sample_cost=22.09375; iteration_number=0;
Metrics - timestamp=1778946941.497345; top_sliced_sample_cost=21.337890625; iteration_number=0;
Metrics - timestamp=1778946941.497345; lowest_sample_cost=21.0; iteration_number=0;
Metrics - timestamp=1778946941.497345; lowest_to_date=21.0; iteration_number=0;
For iteration 10 using the best 80.0 percent of the results
The average cost from the sample is 22.098 and the top-sliced average of the best results is 21.255
The lowest cost from the sample is 21.000, flush=True
The lowest cost to date is 21.000 corresponding to bit string 

In [18]:
sdl_list[0].update_cache_statistics(cost_fn)

The results are saved.  Note, the formulation, hot_start, gradient type and locations are the same for each sub-run so are taken from the last run:

In [19]:
short_title = 'Dist '
full_title = short_title + f' {sdl.formulation} formulation, hot start={sdl.hot_start}, '
full_title = full_title + f'gradient type={sdl.gradient_type}, eta={sdl.eta}, locs={sdl.locations}'
filename = Path.joinpath(datalogger.graph_sub_path,short_title)
sub_title = 'Results with slicing ratio '
x_label = 'Iterations'
if SLICES  == [1]:
    sliced_cost_list_all = None

cost_graph_multi(filename, 
                 SLICES, 
                 sdl_list[0].index_list,
                 av_cost_list_all, 
                 lowest_list_all, 
                 sliced_cost_list_all, 
                 best_dist, 
                 full_title, 
                 sub_title,
                 x_label,
                )